# ML-03 — Frame Your Lane as an ML Task

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/flyrank-bih/flyrank-ml-internship-starter/blob/main/work/notebooks/w02_ml_task_framing.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. My lane as an ML task: ranking / scoring

This is a **ranking / scoring** task because the editor needs to know **which pages first**, not just whether each page belongs to a class. The output will be a priority score that orders pages for review. Ranking matches the real workflow: editorial capacity is limited, so the top of the queue matters more than a perfectly calibrated probability for every page.

In [1]:
from pathlib import Path
import pandas as pd

candidate_paths = [
    Path("data/raw/content_refresh_anonymized.csv"),
    Path("../../data/raw/content_refresh_anonymized.csv"),
]
data_path = next(path for path in candidate_paths if path.exists())
df = pd.read_csv(data_path)

assert df["content_id"].is_unique
assert df["client_id"].nunique() == 32
print(f"Loaded {len(df):,} unique content rows across {df['client_id'].nunique()} clients.")
print("Task type check passed: the unit can be ranked at page level.")

Loaded 30,000 unique content rows across 32 clients.
Task type check passed: the unit can be ranked at page level.


## 2. Target or proxy: a future observed outcome

The eventual target is a page-level **priority score for observed decline in a future time window**. The full warehouse panel should provide the later window: features come from an earlier period, and the label is whether impressions decline in the following period. The starter CSV's `trend_direction` and `trend_pct` are useful for checking prevalence, but they are defined from the same snapshot comparison and must not be used as features or presented as a clean future target. A score is preferable to a hard yes/no proxy because editors need an ordered queue and can choose their review capacity.

In [2]:
label_source = "future observed impression decline"
leakage_columns = {"trend_direction", "trend_pct", "is_declining_label"}
assert leakage_columns.isdisjoint(set(df.columns) - leakage_columns)
assert df["trend_direction"].isin(["new", "flat", "up", "down", "stable"]).all()
print(f"Starter snapshot has {int((df['trend_direction'] == 'down').sum()):,} observed down pages.")
print(f"Reserved from features: {sorted(leakage_columns)}")
print(f"Planned target: {label_source} in a later warehouse window.")

Starter snapshot has 16,262 observed down pages.
Reserved from features: ['is_declining_label', 'trend_direction', 'trend_pct']
Planned target: future observed impression decline in a later warehouse window.


## 3. Success metric: precision@K

The primary metric will be **precision@K**, where K is the number of pages an editor can realistically review in a work period. It answers the operational question directly: among the first K pages in the queue, how many actually decline in the later observed window? I will compare the score with a transparent fixed-rule baseline and report the review capacity and base rate alongside the metric. A useful result is a higher precision@K than the baseline on a client-aware or time-aware holdout; the starter snapshot alone cannot establish that future result.

In [3]:
review_capacity = 100
metric_name = "precision@K"
assert review_capacity > 0
assert metric_name == "precision@K"
print(f"Primary metric: {metric_name}, with K={review_capacity:,} pages.")
print("Comparison required: a fixed-rule baseline evaluated on the same holdout.")

Primary metric: precision@K, with K=100 pages.
Comparison required: a fixed-rule baseline evaluated on the same holdout.


## 4. The unit of analysis, as a real dataframe

The unit is **one pseudonymized content item (page)** at one snapshot or feature-window end. `client_id` is retained for grouping and client-holdout evaluation, but neither `client_id` nor `content_id` is a model feature. The starter slice contains trailing-90-day page metrics and content metadata; it is not a daily panel and therefore cannot by itself create a non-overlapping future label.

In [4]:
id_columns = ["content_id", "client_id"]
assert df["content_id"].nunique() == len(df)
assert df["client_id"].nunique() == 32

print(df[["content_id", "client_id", "content_type", "impressions_90d", "trend_direction"]].head(3).to_string(index=False))
print(f"Shape: {df.shape[0]:,} rows x {df.shape[1]} columns")
print("One row = one pseudonymized content item; IDs are grouping keys, not features.")

          content_id         client_id    content_type  impressions_90d trend_direction
content_304f48230142 client_f369cb89fc keyword article             3803            down
content_a1fb4e703a9e client_4e07408562 keyword article            15320            down
content_9aa793d4d895 client_7f2253d7e2 keyword article            12581            down
Shape: 30,000 rows x 44 columns
One row = one pseudonymized content item; IDs are grouping keys, not features.


## 5. Why ML beats a fixed rule here

A fixed rule is an important baseline, but one threshold cannot express the combination of traffic scale, position, freshness, content type, keyword context, engagement, and missingness. Those signals can interact: a small percentage change on a low-volume page is less actionable than a similar change on a page with meaningful demand. A model may earn its place if it improves precision@K over the rule on an honest holdout. If it does not, the transparent rule or a dashboard is the better decision tool.

In [5]:
candidate_feature_columns = [
    column for column in df.columns
    if column not in {"content_id", "client_id", "trend_direction", "trend_pct", "is_declining_label"}
]
assert "content_id" not in candidate_feature_columns
assert "client_id" not in candidate_feature_columns
assert "trend_direction" not in candidate_feature_columns
assert "trend_pct" not in candidate_feature_columns
print(f"Candidate signal columns after ID and leakage exclusions: {len(candidate_feature_columns)}")
print(f"Missing search volume: {df['search_volume'].isna().sum():,}; missing word count: {df['word_count'].isna().sum():,}.")
print("Missingness will be handled explicitly rather than blindly filled with zero.")

Candidate signal columns after ID and leakage exclusions: 40
Missing search volume: 2,468; missing word count: 7,699.
Missingness will be handled explicitly rather than blindly filled with zero.


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.